In [64]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Type, List


def conv3x3(in_planes: int, out_planes: int, stride: int = 1) -> nn.Conv2d:
    """3x3 convolution with padding."""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride, padding=1, bias=False)


class LambdaLayer(nn.Module):
    """Implements Option A Downsampling."""
    def __init__(self, lambd):
        super(LambdaLayer, self).__init__()
        self.lambd = lambd

    def forward(self, x):
        return self.lambd(x)


class BasicBlock(nn.Module):
    expansion: int = 1

    def __init__(self, inplanes: int, planes: int, stride: int = 1, downsample: nn.Module = None):
        super().__init__()
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)

        self.downsample = downsample




    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        if self.downsample is not None:
            identity = self.downsample(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out


class ResNet(nn.Module):
    def __init__(self, block: Type[BasicBlock], layers: List[int], num_classes: int = 1000):
        super().__init__()
        self.inplanes = 16

        # Initial 3x3 Conv Layer (No max pooling for CIFAR)
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        # Residual layers: 16 → 32 → 64 channels
        self.layer1 = self._make_layer(block, 16, layers[0], stride=1)
        self.layer2 = self._make_layer(block, 32, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 64, layers[2], stride=2)

        # Global Average Pooling + Fully Connected Layer
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc =nn.Sequential(
                #  nn.Dropout(p=0.3),
                 nn.Linear(64 * block.expansion, num_classes))

        # Weight Initialization
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)



    def _make_layer(self, block: Type[BasicBlock], planes: int, blocks: int, stride: int = 1):
        """Create a ResNet layer with multiple residual blocks."""
        downsample = None
        if stride != 1 or self.inplanes != planes * block.expansion:
            # downsample = LambdaLayer(lambda x:
            #                          F.pad(x[:, :, ::2, ::2], (0, 0, 0, 0, planes // 4, planes // 4), "constant", 0))

            downsample = nn.Sequential(
            nn.Conv2d(self.inplanes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
            nn.BatchNorm2d(planes * block.expansion)
            )


        layers = [block(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


# ResNet Model Definitions for CIFAR-100
class_num =100
def resnet20(num_classes: int = class_num ) -> ResNet:
    """ResNet-20: 3 residual blocks per stage."""
    return ResNet(BasicBlock, [3, 3, 3], num_classes)


def resnet56(num_classes: int = class_num ) -> ResNet:
    """ResNet-56: 9 residual blocks per stage."""
    return ResNet(BasicBlock, [9, 9, 9], num_classes)


def resnet110(num_classes: int = class_num ) -> ResNet:
    """ResNet-110: 18 residual blocks per stage."""
    return ResNet(BasicBlock, [18, 18, 18], num_classes)


In [65]:


import torchvision
import torchvision.transforms as transforms

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Training Transformations for CIFAR-100
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=10),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
])

# Test Transformations for CIFAR-100
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
])

# Load CIFAR-100 dataset
train_dataset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, drop_last=False)

test_dataset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4, drop_last=False)


Files already downloaded and verified
Files already downloaded and verified


In [66]:
teacher_model = resnet56(num_classes=100).to(device)
student_model = resnet20(num_classes=100).to(device)
teacher_model.load_state_dict(torch.load("resnet56_cifar100_best.pth", weights_only=True), strict=False)

teacher_model.eval()

ResNet(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=Fals

In [67]:
# from torchmetrics.classification import Accuracy

# top1_acc = Accuracy(task="multiclass", num_classes=100, top_k=1).to(device)
# top5_acc = Accuracy(task="multiclass", num_classes=100, top_k=5).to(device)
# # Evaluate Top-1 and Top-5 Accuracy
# top1, top5 = 0.0, 0.0
# total_samples = 0

# with torch.no_grad():
#     for images, labels in test_loader:
#         images, labels = images.to(device), labels.to(device)

#         outputs = teacher_model(images)  # Get model predictions

#         top1 += top1_acc(outputs, labels) * images.size(0)
#         top5 += top5_acc(outputs, labels) * images.size(0)
#         total_samples += images.size(0)

# # Compute final accuracy
# top1_accuracy = top1 / total_samples * 100
# top5_accuracy = top5 / total_samples * 100

# print(f"Top-1 Accuracy: {top1_accuracy:.2f}%")
# print(f"Top-5 Accuracy: {top5_accuracy:.2f}%")


In [68]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Literal

LossType = Literal['mse','l1','kl','hinge']

# class CBAMFeatureLoss(nn.Module):
#     """
#     Compute loss between two CBAM‐modulated feature maps A_t, A_s ∈ ℝ^{B×C×H×W}.

#     kind = 'mse'   : MSE(A_s, A_t)
#          = 'l1'    : L1(A_s, A_t)
#          = 'kl'    : KL(softmax(A_t) || softmax(A_s))
#          = 'hinge' : mean( ReLU(|A_s - A_t| - margin) )
#     """
#     def __init__(self,
#                  kind:    LossType = 'mse',
#                  margin:  float    = 0.15,
#                  temp:    float    = 1.0):
#         super().__init__()
#         assert kind in ('mse','l1','kl','hinge')
#         self.kind   = kind
#         self.margin = margin
#         self.temp   = temp

#     def forward(self, A_t: torch.Tensor, A_s: torch.Tensor) -> torch.Tensor:
#         B, C, H, W = A_t.shape
#         # flatten each map to a vector of length C*H*W
#         t = A_t.view(B, -1)
#         s = A_s.view(B, -1)

#         if self.kind == 'mse':
#             # pointwise MSE
#             return F.mse_loss(s, t)

#         elif self.kind == 'l1':
#             # pointwise MAE
#             return F.l1_loss(s, t)

#         elif self.kind == 'kl':
#             # treat each flattened map as a distribution
#             # KL( P_t || P_s ) = sum P_t log(P_t / P_s)
#             P_t = F.softmax(t / self.temp, dim=-1)
#             log_P_s = F.log_softmax(s / self.temp, dim=-1)
#             # reduction='batchmean' gives (1/B) ∑_b ∑_i P_t[b,i] * (log P_t[b,i] - log P_s[b,i])
#             return F.kl_div(log_P_s, P_t, reduction='batchmean')

#         else:  # 'hinge'
#             # margin‐ReLU on absolute difference
#             d = torch.abs(s - t)
#             m = F.relu(d - self.margin)
#             return m.mean()
class CBAMFeatureLoss(nn.Module):
    def __init__(self,
                 kind: str = 'mse',  # 'mse', 'l1', 'kl', 'hinge'
                 margin: float = 0.15,
                 temp: float = 1.0):
        super().__init__()
        assert kind in ('mse', 'l1', 'kl', 'hinge')
        self.kind = kind
        self.margin = margin
        self.temp = temp

    def pairwise_loss(self, A_t, A_s):
        # A_t and A_s: shape (B, C, H, W)
        B = A_t.size(0)
        t = A_t.view(B, -1)
        s = A_s.view(B, -1)

        if self.kind == 'mse':
            return F.mse_loss(s, t, reduction='none').mean(dim=1)  # (B,)
        elif self.kind == 'l1':
            return F.l1_loss(s, t, reduction='none').mean(dim=1)  # (B,)
        elif self.kind == 'kl':
            P_t = F.softmax(t / self.temp, dim=-1)
            log_P_s = F.log_softmax(s / self.temp, dim=-1)
            return F.kl_div(log_P_s, P_t, reduction='none').sum(dim=1)  # (B,)
        else:  # hinge
            d = torch.abs(s - t)
            m = F.relu(d - self.margin)
            return m.mean(dim=1)  # (B,)

    def forward(self, teacher_feat, student_feat, alpha_matrix):
        """
        teacher_feat, student_feat: Single pair of CBAM features (B, C, H, W)
        alpha_matrix: Tensor (B, 1, 1) — scalar attention weight per sample
        """
        loss_per_sample = self.pairwise_loss(teacher_feat, student_feat)  # (B,)
        weights = alpha_matrix.view(-1)  # (B,)
        weighted_loss = (loss_per_sample * weights).mean()
        return weighted_loss


In [69]:
# class FocalLoss(nn.Module):
#     def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
#         super().__init__()
#         self.gamma = gamma
#         self.alpha = alpha  # Tensor of shape [num_classes] or None
#         self.reduction = reduction

#     def forward(self, logits, targets):
#         ce_loss = F.cross_entropy(logits, targets, reduction='none')
#         pt = torch.exp(-ce_loss)
#         focal_loss = ((1 - pt) ** self.gamma) * ce_loss

#         if self.alpha is not None:
#             alpha_t = self.alpha[targets]
#             focal_loss = alpha_t * focal_loss

#         return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()


In [70]:
# class LayerWiseAttention(nn.Module):
#     def __init__(self, num_layers):
#         super().__init__()
#         self.weights = nn.Parameter(torch.ones(num_layers))

#     def forward(self):
#         return F.softmax(self.weights, dim=0)


In [71]:
def extract_cbam_vector(cbam_feat: torch.Tensor):
    # cbam_feat shape: (B, C, H, W)
    B = cbam_feat.shape[0]
    return cbam_feat.mean(dim=[2, 3])  # Global average over H, W → shape: (B, C)


In [72]:
def compute_alpha_matrix(student_feat, teacher_feat, temperature=1.0):
    """
    student_feat, teacher_feat: CBAM-processed feature maps (B, C, H, W)
    Return: α (B, 1, 1) — scalar attention weight per sample
    """
    s_vec = extract_cbam_vector(student_feat)  # (B, C)
    t_vec = extract_cbam_vector(teacher_feat)  # (B, C)

    # Compute scaled dot product similarity per sample
    logits = (s_vec * t_vec).sum(dim=1, keepdim=True) / temperature  # (B, 1)

    # Optionally apply softmax or sigmoid (here: sigmoid gives scalar between 0–1)
    alpha = torch.sigmoid(logits)  # (B, 1)

    return alpha.unsqueeze(1)  # Final shape: (B, 1, 1)


In [73]:


class MultiLayerDistillationLoss(nn.Module):
    def __init__(self, teacher_channels,
                 alpha=0.6, temperature=3.0, beta=0.2,
                 pool_size=4, distance_type='smoothl1',temp=1,
                 teacher_detach=True, align_corners=False):
        super().__init__()

        self.alpha = alpha
        self.temperature = temperature
        self.beta = beta
        self.ce = nn.CrossEntropyLoss(label_smoothing=0.1)
        self.pool_size = pool_size
        self.distance_type = distance_type
        self.teacher_detach = teacher_detach
        self.temp=temp

        
        self.log_sigma_ce = nn.Parameter(torch.tensor(0.0))
        self.log_sigma_kl = nn.Parameter(torch.tensor(0.0))
        self.log_sigma_feat = nn.Parameter(torch.tensor(0.0))
        self.loss_fn = CBAMFeatureLoss(kind=self.distance_type)

    def forward(self, teacher_feats, student_feats, student_logits, teacher_logits, labels):
        assert len(teacher_feats) == len(student_feats), "Mismatch in feature layers"
        
        # weights = self.layer_attention()  # softmax: [L]

        # Feature distillation with layer-wise weighting
        feature_loss = 0.0
        for i, (F_t, F_s) in enumerate(zip(teacher_feats, student_feats)):
            alpha_matrix = compute_alpha_matrix(F_t, F_s,temperature=self.temp)  # (B, 3, 3)
            loss_i = self.loss_fn(F_t, F_s, alpha_matrix)
            feature_loss +=  loss_i
        feature_loss /= len(teacher_feats)
        # KL divergence for logit distillation
        T = self.temperature
        teacher_soft = F.softmax(teacher_logits / T, dim=1)
        student_log_soft = F.log_softmax(student_logits / T, dim=1)
        kd_loss = F.kl_div(student_log_soft, teacher_soft, reduction='batchmean') * (T ** 2)

        # Handle hard labels (possibly from mixup)
        if labels.dim() == 2:
            labels = labels.argmax(dim=1)

        ce_loss = self.ce(student_logits, labels)

        # Combine losses
        total_loss = self.alpha * kd_loss + 0.5 * feature_loss + self.beta * ce_loss
        return total_loss


In [74]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Type, List

class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        """
        Channel attention module.
        Uses both average and max pooling to capture channel-wise importance.
        """
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        """
        Spatial attention module.
        Aggregates information along the channel dimension using average and max pooling,
        then applies a convolution to generate a spatial attention map.
        """
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        attention = self.conv(x_cat)
        return self.sigmoid(attention)

class CBAM(nn.Module):
    def __init__(self, in_channels, reduction=16, spatial_kernel=7):
        """
        CBAM module: sequentially applies channel attention and spatial attention.
        """
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention(spatial_kernel)

    def forward(self, x):
        x_out = x * self.channel_attention(x)
        x_out = x_out * self.spatial_attention(x_out)
        return x_out

In [75]:
student_model.load_state_dict(torch.load("resnet20_kd_best.pth", weights_only=True), strict=False )


<All keys matched successfully>

In [76]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision.models.feature_extraction import create_feature_extractor

from tqdm import tqdm
import torch.optim as optim

kd_criterion = MultiLayerDistillationLoss(teacher_channels=[16, 32, 64],
    temperature=4.0,
    alpha=0.5,
    beta=0.4,
    pool_size=4,
    distance_type='mse',
    teacher_detach=True).to(device)

# Create feature extractors once
return_nodes = {
    'layer1': 'feat1',
    'layer2': 'feat2',
    'layer3': 'feat3'
}
teacher_model.eval()
teacher_extractor = create_feature_extractor(teacher_model, return_nodes=return_nodes)
student_extractor = create_feature_extractor(student_kd_model, return_nodes=return_nodes)

cbam1 = CBAM(in_channels=16).to(device)
cbam2 = CBAM(in_channels=32).to(device)
cbam3 = CBAM(in_channels=64).to(device)

# Optimizer and Scheduler
max_epochs = 400
# optimizer = optim.Adam(student_model.parameters(), lr=0.01)
optimizer = optim.SGD(student_kd_model.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4, nesterov = True)
scheduler = CosineAnnealingLR(optimizer, T_max=max_epochs, eta_min = 1e-5)

# Early Stopping Variables
patience = 25  # epochs of no improvement allowed
patience_counter = 0
best_val_accuracy = 0.0
model_save_path = "resnet20_kd_best.pth"

# Training Loop
for epoch in range(max_epochs):
    student_kd_model.train()
    epoch_loss = 0.0
    step = 0
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs} Training", leave=False)

    for images, labels in train_bar:
        step += 1
        images, labels = images.to(device), labels.to(device)
        # images, labels = mixup_fn ( images, labels)
        optimizer.zero_grad()

        # Teacher forward (no grad)
        with torch.no_grad():
            teacher_feats_raw = teacher_extractor(images)
            teacher_feats = [
            cbam1(teacher_feats_raw['feat1']),
            cbam2(teacher_feats_raw['feat2']),
            cbam3(teacher_feats_raw['feat3']),
            ]
            teacher_logits = teacher_model(images)

        student_feats_raw = student_extractor(images)
        student_feats = [
        cbam1(student_feats_raw['feat1']),
        cbam2(student_feats_raw['feat2']),
        cbam3(student_feats_raw['feat3']),
        ]
        student_logits = student_kd_model(images)

        # student_feats = student_extractor(images)


        # Compute KD loss (feature-based + logit KD)
        loss = kd_criterion(teacher_feats, student_feats, teacher_logits, student_logits, labels)
        loss.backward()
        # Clip gradients
        # if epoch >= 40:
        #     mixup_fn.cutmix_alpha = 1.0
        #     mixup_fn.prob         = 0.8
        #     torch.nn.utils.clip_grad_norm_(student_model.parameters(), max_norm=5.0)
        torch.nn.utils.clip_grad_norm_(student_kd_model.parameters(), max_norm=5.0)
        optimizer.step()

        epoch_loss += loss.item()
        train_bar.set_postfix({"Batch Loss": loss.item()})

    scheduler.step()
    epoch_loss /= step
    print(f"Epoch {epoch+1} Training Loss: {epoch_loss:.4f}")

    # Validation Phase every 'val_interval' epochs (or every epoch if you wish)
    if (epoch + 1) % 5 == 0:
        student_kd_model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        val_steps = 0
        val_bar = tqdm(test_loader, desc=f"Epoch {epoch+1} Validation", leave=False)
        with torch.no_grad():
            for images, labels in val_bar:
                val_steps += 1
                images, labels = images.to(device), labels.to(device)

                # Use the same feature extractors for teacher & student
                teacher_out = teacher_extractor(images)
                teacher_feats = [
                cbam1(teacher_feats_raw['feat1']),
                cbam2(teacher_feats_raw['feat2']),
                cbam3(teacher_feats_raw['feat3']),
                ]
                teacher_logits = teacher_model(images)

                student_out = student_extractor(images)
                student_feats = [
                cbam1(student_feats_raw['feat1']),
                cbam2(student_feats_raw['feat2']),
                cbam3(student_feats_raw['feat3']),
                ]
                student_logits = student_kd_model(images)

                loss = kd_criterion(teacher_feats, student_feats, teacher_logits, student_logits, labels)
                val_loss += loss.item()

                _, predicted = torch.max(student_logits, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= val_steps
        val_accuracy = correct / total
        print(f"Epoch {epoch+1} Validation Loss: {val_loss:.4f}")
        print(f"Epoch {epoch+1} Validation Accuracy: {val_accuracy:.4f}")

        # Early Stopping & Model Saving
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            patience_counter = 0  # Reset patience counter
            torch.save(student_kd_model.state_dict(), model_save_path)
            print(f"New best model saved at epoch {epoch+1} with Validation Accuracy: {val_accuracy:.4f}")
        else:
            patience_counter += 1
            print(f"Early stopping patience: {patience_counter}/{patience}")

        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

print("Training Complete!")

Epoch 1 Training Loss: 1.7067


Epoch 2 Training Loss: 1.6347


Epoch 3 Training Loss: 1.5933


Epoch 4 Training Loss: 1.5590


Epoch 5 Training Loss: 1.5331


...

Epoch 8 Training Loss: 1.4691


Epoch 9 Training Loss: 1.4526


Epoch 10 Training Loss: 1.4380


Epoch 10 Validation Loss: 1.7133
Epoch 10 Validation Accuracy: 0.0628
Early stopping patience: 1/25


KeyboardInterrupt: 